# Turbidity Within-1-Day Feature and Model Comparison

Goal: compare whether curated context features improve Turbidity prediction over curated Sentinel-2 features under a clean temporal match condition.

Main setup:
- Dataset: `relaxed_v4`
- Target: `turbidity_NTU_clean`
- Match condition: `abs_date_diff_days <= 1`
- No outlier removal
- Train: BE 2562-2566
- Test: BE 2567-2568
- Train target: `log1p(Turbidity)`
- Metrics reported on both log scale and original NTU scale


In [ ]:
# Imports
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, ElasticNet, BayesianRidge, HuberRegressor
from sklearn.svm import LinearSVR
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_row", 90)
pd.set_option("display.width", 160)


In [ ]:
# Config
DATA_PATH = Path(r".\Data\tabular_data_14\train\v4\rswq_model_input_relaxed_s2_context_2562_2568.csv")
TARGET_COL = "turbidity_NTU_clean"
TARGET_LOG_COL = "turbidity_log1p"
RANDOM_STATE = 42

TRAIN_YEAR_MIN = 2562
TRAIN_YEAR_MAX = 2566
TEST_YEAR_MIN = 2567
TEST_YEAR_MAX = 2568
MAX_MATCH_DAYS = 1

S2_SELECTED = [
    "B1_median", "B2_median", "B3_median", "B4_median", "B5_median", "B6_median",
    "B7_median", "B8_median", "B8A_median", "B9_median", "B11_median", "B12_median",
    "NDTI_median", "NDWI_median",
]

CONTEXT_SELECTED = [
    "rain_gsmap_prior_24h_mm",
    "era5h_runoff_prior_24h_mm",
    "era5h_runoff_prior_1h_mm",
    "wc_water_frac_5000m",
]

FEATURE_SETS = {
    "S2 selected": S2_SELECTED,
    "Context selected": CONTEXT_SELECTED,
    "S2 selected + Context selected": S2_SELECTED + CONTEXT_SELECTED,
}


In [ ]:
# Load data and filter within 1 day.
df = pd.read_csv(DATA_PATH, low_memory=False)
df[TARGET_LOG_COL] = np.log1p(pd.to_numeric(df[TARGET_COL], errors="coerce"))

all_features = sorted(set(sum(FEATURE_SETS.values(), [])))
missing_features = [c for c in all_features if c not in df.columns]
if missing_features:
    raise KeyError(f"Missing feature columns: {missing_features}")

match_days = pd.to_numeric(df["abs_date_diff_days"], errors="coerce")
year = pd.to_numeric(df["year_be"], errors="coerce")
base_mask = df[TARGET_COL].notna() & match_days.le(MAX_MATCH_DAYS)

within1_df = df.loc[base_mask].copy()
train_df = df.loc[base_mask & year.between(TRAIN_YEAR_MIN, TRAIN_YEAR_MAX)].copy()
test_df = df.loc[base_mask & year.between(TEST_YEAR_MIN, TEST_YEAR_MAX)].copy()

split_summary = pd.DataFrame({
    "split": ["within1_all", "train_2562_2566", "test_2567_2568"],
    "rows": [len(within1_df), len(train_df), len(test_df)],
    "stations": [
        within1_df["station_canonical"].nunique(),
        train_df["station_canonical"].nunique(),
        test_df["station_canonical"].nunique(),
    ],
    "target_median": [
        within1_df[TARGET_COL].median(),
        train_df[TARGET_COL].median(),
        test_df[TARGET_COL].median(),
    ],
    "target_p95": [
        within1_df[TARGET_COL].quantile(0.95),
        train_df[TARGET_COL].quantile(0.95),
        test_df[TARGET_COL].quantile(0.95),
    ],
    "target_max": [
        within1_df[TARGET_COL].max(),
        train_df[TARGET_COL].max(),
        test_df[TARGET_COL].max(),
    ],
})

display(split_summary.round(4))

feature_summary = pd.DataFrame({
    "feature_set": list(FEATURE_SETS.keys()),
    "n_features": [len(v) for v in FEATURE_SETS.values()],
    "features": [", ".join(v) for v in FEATURE_SETS.values()],
})
display(feature_summary)


In [ ]:
# Optional model imports.
try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception as exc:
    HAS_XGBOOST = False
    print("XGBoost not available:", exc)

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except Exception as exc:
    HAS_CATBOOST = False
    print("CatBoost not available:", exc)


def make_models(random_state=42):
    models = {
        "DummyMedian": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DummyRegressor(strategy="median")),
        ]),
        "Ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]),
        "ElasticNet": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", ElasticNet(alpha=0.01, l1_ratio=0.15, max_iter=20000, random_state=random_state)),
        ]),
        "BayesianRidge": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", BayesianRidge()),
        ]),
        "HuberRegressor": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", HuberRegressor(epsilon=1.35, alpha=0.0001, max_iter=2000)),
        ]),
        "RandomForest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=random_state, n_jobs=-1)),
        ]),
        "ExtraTrees": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesRegressor(n_estimators=500, min_samples_leaf=2, random_state=random_state, n_jobs=-1)),
        ]),
        "HistGradientBoosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingRegressor(max_iter=300, learning_rate=0.03, l2_regularization=0.1, random_state=random_state)),
        ]),
    }

    for c in [0.3, 0.5, 1.0, 2.0, 5.0]:
        for epsilon in [0.0, 0.05, 0.10]:
            models[f"LinearSVR_C{c}_eps{epsilon}"] = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LinearSVR(C=c, epsilon=epsilon, random_state=random_state, max_iter=30000)),
            ])

    for n_components in [2]:
        models[f"PLS_{n_components}"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", PLSRegression(n_components=n_components)),
        ])

    if HAS_XGBOOST:
        models["XGBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(
                n_estimators=500,
                max_depth=2,
                learning_rate=0.03,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_lambda=5.0,
                objective="reg:squarederror",
                random_state=random_state,
                n_jobs=-1,
            )),
        ])

    if HAS_CATBOOST:
        models["CatBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", CatBoostRegressor(
                iterations=700,
                depth=3,
                learning_rate=0.03,
                loss_function="RMSE",
                random_seed=random_state,
                verbose=False,
                allow_writing_files=False,
            )),
        ])

    return models


In [ ]:
# Metric helpers.
def safe_corr(corr_func, y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ok = np.isfinite(y_true) & np.isfinite(y_pred)
    if ok.sum() < 3 or np.std(y_true[ok]) == 0 or np.std(y_pred[ok]) == 0:
        return np.nan
    return float(corr_func(y_true[ok], y_pred[ok])[0])


def regression_metrics(y_true_log, pred_log, y_true_original):
    pred_log = np.asarray(pred_log, dtype=float).ravel()
    pred_log_clipped = np.clip(pred_log, 0, None)
    pred_original = np.expm1(pred_log_clipped)
    y_true_original = np.asarray(y_true_original, dtype=float)

    return {
        "log_r2": float(r2_score(y_true_log, pred_log)),
        "log_rmse": float(np.sqrt(mean_squared_error(y_true_log, pred_log))),
        "original_r2": float(r2_score(y_true_original, pred_original)),
        "original_rmse": float(np.sqrt(mean_squared_error(y_true_original, pred_original))),
        "original_mae": float(mean_absolute_error(y_true_original, pred_original)),
        "spearman_r": safe_corr(spearmanr, y_true_original, pred_original),
        "pearson_r": safe_corr(pearsonr, y_true_original, pred_original),
        "bias": float(np.mean(pred_original - y_true_original)),
    }


In [ ]:
# Train/evaluate all feature-set x model combinations.
models = make_models(RANDOM_STATE)
rows = []
prediction_tables = {}

for feature_set_name, feature_cols in FEATURE_SETS.items():
    X_train = train_df[feature_cols].copy()
    y_train = train_df[TARGET_LOG_COL].copy()
    X_test = test_df[feature_cols].copy()
    y_test = test_df[TARGET_LOG_COL].copy()

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        pred_train_log = np.asarray(model.predict(X_train)).ravel()
        pred_test_log = np.asarray(model.predict(X_test)).ravel()

        train_metrics = regression_metrics(y_train, pred_train_log, train_df[TARGET_COL])
        test_metrics = regression_metrics(y_test, pred_test_log, test_df[TARGET_COL])

        row = {
            "feature_set": feature_set_name,
            "model": model_name,
            "n_features": len(feature_cols),
            "n_train": len(train_df),
            "n_test": len(test_df),
        }
        row.update({f"train_{k}": v for k, v in train_metrics.items()})
        row.update({f"test_{k}": v for k, v in test_metrics.items()})
        rows.append(row)

        prediction_tables[(feature_set_name, model_name)] = pd.DataFrame({
            "sample_id": test_df["sample_id"].values,
            "station_canonical": test_df["station_canonical"].values,
            "year_be": test_df["year_be"].values,
            "observed": test_df[TARGET_COL].values,
            "predicted": np.expm1(np.clip(pred_test_log, 0, None)),
            "observed_log": y_test.values,
            "predicted_log": pred_test_log,
        })

results = pd.DataFrame(rows).sort_values(
    ["test_original_r2", "test_log_r2"],
    ascending=False,
).reset_index(drop=True)

show_cols = [
    "feature_set", "model", "n_features", "n_train", "n_test",
    "test_original_r2", "test_original_rmse", "test_original_mae",
    "test_log_r2", "test_log_rmse", "test_spearman_r", "test_pearson_r", "test_bias",
    "train_original_r2", "train_log_r2",
]

display(results[show_cols].round(4))


In [ ]:
# Best model by feature set.
best_by_feature = (
    results.sort_values(["feature_set", "test_original_r2", "test_log_r2"], ascending=[True, False, False])
    .groupby("feature_set", as_index=False)
    .head(1)
    .sort_values("test_original_r2", ascending=False)
    .reset_index(drop=True)
)

display(best_by_feature[show_cols].round(4))

# Compare context gain against S2 selected.
s2_best = best_by_feature.loc[best_by_feature["feature_set"].eq("S2 selected"), "test_original_r2"]
s2_context_best = best_by_feature.loc[best_by_feature["feature_set"].eq("S2 selected + Context selected"), "test_original_r2"]
if len(s2_best) and len(s2_context_best):
    print("Context gain over S2 selected (original R2):", round(float(s2_context_best.iloc[0] - s2_best.iloc[0]), 4))


In [ ]:
# Plot observed vs predicted for each feature-set best model.
for _, row in best_by_feature.iterrows():
    key = (row["feature_set"], row["model"])
    pred = prediction_tables[key].copy()
    pred["residual"] = pred["predicted"] - pred["observed"]

    print(f"Best for {row['feature_set']}: {row['model']} | R2={row['test_original_r2']:.3f} | RMSE={row['test_original_rmse']:.2f}")
    display(pred.sort_values("observed", ascending=False).head(8).round(4))

    fig, ax = plt.subplots(figsize=(5.2, 4.8))
    ax.scatter(pred["observed"], pred["predicted"], alpha=0.8)
    limit = max(pred["observed"].max(), pred["predicted"].max()) * 1.05
    ax.plot([0, limit], [0, limit], linestyle="--", color="gray", linewidth=1)
    ax.set_xlim(0, limit)
    ax.set_ylim(0, limit)
    ax.set_xlabel("Observed Turbidity (NTU)")
    ax.set_ylabel("Predicted Turbidity (NTU)")
    ax.set_title(f"{row['feature_set']} | {row['model']}")
    ax.grid(True, alpha=0.3)
    plt.show()


## Interpretation Notes

Use `S2 selected` as the main Sentinel-2-only baseline. The original fixed context set contains 24-hour rainfall, 1- and 24-hour runoff, and 5-km WorldCover water fraction. It does not contain wind or season features.

Because this experiment uses only within-1-day matchups, the test set is small. Report `n_train`, `n_test`, and avoid overclaiming a single high R2 value without the sample-size context. The model-specific context experiment at the end of the notebook performs feature selection using training years only.


## LinearSVR Explainability for the Locked Turbidity Model

This section explains the previously locked `LinearSVR_C0.5_eps0.0` model with the fixed `S2 selected + Context selected` feature set. It is retained as the original artifact candidate and is not used to select context features for the other model families.

The model is trained on `log1p(Turbidity)`, so coefficient and SHAP values below are on the log scale. Positive values push predicted turbidity upward; negative values push predicted turbidity downward.


In [ ]:
# Fit the final best LinearSVR model for explanation.
import warnings
warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", category=UserWarning, module="tqdm.auto")

try:
    import shap
except ImportError as exc:
    raise ImportError(
        "SHAP is not installed in this kernel. Install it with: python -m pip install shap"
    ) from exc

LINEAR_EXPLAIN_FEATURE_SET_NAME = "S2 selected + Context selected"
LINEAR_EXPLAIN_MODEL_NAME = "LinearSVR_C0.5_eps0.0"
LINEAR_EXPLAIN_SPLIT = "test"

if LINEAR_EXPLAIN_FEATURE_SET_NAME not in FEATURE_SETS:
    raise KeyError(f"Unknown feature set: {LINEAR_EXPLAIN_FEATURE_SET_NAME}")

linear_models = make_models(RANDOM_STATE)
if LINEAR_EXPLAIN_MODEL_NAME not in linear_models:
    raise KeyError(f"Unknown model: {LINEAR_EXPLAIN_MODEL_NAME}")

linear_feature_cols = FEATURE_SETS[LINEAR_EXPLAIN_FEATURE_SET_NAME]
linear_pipeline = linear_models[LINEAR_EXPLAIN_MODEL_NAME]
linear_pipeline.fit(train_df[linear_feature_cols], train_df[TARGET_LOG_COL])

X_train_linear_raw = train_df[linear_feature_cols].copy()
X_test_linear_raw = test_df[linear_feature_cols].copy()
X_explain_linear_raw = X_test_linear_raw if LINEAR_EXPLAIN_SPLIT == "test" else X_train_linear_raw
explain_linear_df = test_df if LINEAR_EXPLAIN_SPLIT == "test" else train_df

linear_preprocess = Pipeline(linear_pipeline.steps[:-1])
linear_final_model = linear_pipeline.steps[-1][1]

X_train_linear = pd.DataFrame(
    linear_preprocess.transform(X_train_linear_raw),
    columns=linear_feature_cols,
    index=X_train_linear_raw.index,
)
X_explain_linear = pd.DataFrame(
    linear_preprocess.transform(X_explain_linear_raw),
    columns=linear_feature_cols,
    index=X_explain_linear_raw.index,
)

pred_test_log = np.asarray(linear_pipeline.predict(X_test_linear_raw)).ravel()
linear_metrics = regression_metrics(test_df[TARGET_LOG_COL], pred_test_log, test_df[TARGET_COL])
linear_pred_original = np.expm1(np.clip(pred_test_log, 0, None))

linear_summary = pd.DataFrame([{
    "feature_set": LINEAR_EXPLAIN_FEATURE_SET_NAME,
    "model": LINEAR_EXPLAIN_MODEL_NAME,
    "explained_split": LINEAR_EXPLAIN_SPLIT,
    "n_features": len(linear_feature_cols),
    "n_train": len(train_df),
    "n_test": len(test_df),
    "test_original_r2": linear_metrics["original_r2"],
    "test_original_rmse": linear_metrics["original_rmse"],
    "test_log_r2": linear_metrics["log_r2"],
    "test_log_rmse": linear_metrics["log_rmse"],
    "test_spearman_r": linear_metrics["spearman_r"],
    "test_bias": linear_metrics["bias"],
}])

display(linear_summary.round(4))


In [ ]:
# LinearSVR standardized coefficient interpretation.
linear_coef = np.asarray(linear_final_model.coef_, dtype=float).ravel()
linear_intercept = float(np.asarray(linear_final_model.intercept_).ravel()[0])

coef_table = (
    pd.DataFrame({
        "feature": linear_feature_cols,
        "coefficient_log1p_standardized": linear_coef,
        "abs_coefficient": np.abs(linear_coef),
        "direction": np.where(linear_coef >= 0, "pushes prediction up", "pushes prediction down"),
    })
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

print(f"LinearSVR equation on standardized features: log1p(Turbidity) = {linear_intercept:.4f} + sum(coef_j * z_j)")
print("z_j = median-imputed and standardized feature value.")
display(coef_table.round(5))

fig, ax = plt.subplots(figsize=(7, 5))
plot_df = coef_table.sort_values("coefficient_log1p_standardized")
colors = np.where(plot_df["coefficient_log1p_standardized"] >= 0, "#4C78A8", "#E45756")
ax.barh(plot_df["feature"], plot_df["coefficient_log1p_standardized"], color=colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("LinearSVR coefficient on log1p(Turbidity), standardized features")
ax.set_title(f"LinearSVR coefficients | {LINEAR_EXPLAIN_FEATURE_SET_NAME}")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# Linear SHAP for the final LinearSVR model.
# This decomposes the LinearSVR prediction into feature contributions on log1p(Turbidity).
linear_shap_explainer = shap.LinearExplainer(linear_final_model, X_train_linear)
linear_shap_explanation = linear_shap_explainer(X_explain_linear)
linear_shap_values = np.asarray(linear_shap_explanation.values)

linear_shap_importance = (
    pd.DataFrame({
        "feature": linear_feature_cols,
        "coefficient_log1p_standardized": linear_coef,
        "mean_abs_shap_log1p": np.abs(linear_shap_values).mean(axis=0),
        "mean_shap_log1p": linear_shap_values.mean(axis=0),
    })
    .sort_values("mean_abs_shap_log1p", ascending=False)
    .reset_index(drop=True)
)

print("Linear SHAP importance for the final LinearSVR model.")
print("Larger mean_abs_shap_log1p = stronger average contribution to log1p(Turbidity).")
display(linear_shap_importance.round(5))

fig, ax = plt.subplots(figsize=(7, 5))
plot_df = linear_shap_importance.head(20).sort_values("mean_abs_shap_log1p")
ax.barh(plot_df["feature"], plot_df["mean_abs_shap_log1p"], color="#4C78A8")
ax.set_xlabel("Mean |SHAP value| on log1p(Turbidity)")
ax.set_title(f"Linear SHAP global importance | {LINEAR_EXPLAIN_MODEL_NAME}")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

shap.summary_plot(
    linear_shap_values,
    X_explain_linear,
    feature_names=linear_feature_cols,
    max_display=min(20, len(linear_feature_cols)),
    show=False,
)
plt.title(f"Linear SHAP summary | {LINEAR_EXPLAIN_MODEL_NAME}")
plt.tight_layout()
plt.show()


In [ ]:
# Local Linear SHAP explanation for the highest-observed turbidity sample in the explained split.
pred_log_explain = np.asarray(linear_pipeline.predict(X_explain_linear_raw)).ravel()
pred_original_explain = np.expm1(np.clip(pred_log_explain, 0, None))

local_linear_meta = explain_linear_df.loc[X_explain_linear_raw.index, [
    c for c in ["sample_id", "station_canonical", "year_be", "field_date", "sat_date", "abs_date_diff_days", TARGET_COL]
    if c in explain_linear_df.columns
]].copy()
local_linear_meta["predicted_turbidity"] = pred_original_explain
local_linear_meta["residual"] = local_linear_meta["predicted_turbidity"] - local_linear_meta[TARGET_COL]

sample_pos = int(np.nanargmax(local_linear_meta[TARGET_COL].to_numpy(dtype=float)))
print("Local LinearSVR SHAP example: highest observed turbidity in explained split")
display(local_linear_meta.iloc[[sample_pos]].round(4))

shap.plots.waterfall(linear_shap_explanation[sample_pos], max_display=min(12, len(linear_feature_cols)), show=False)
plt.title("Local LinearSVR SHAP waterfall on log1p(Turbidity)")
plt.tight_layout()
plt.show()


## Original Fixed S2+Context Explainability

This section preserves the explanation workflow for the original fixed four-feature context experiment. The explained model is selected from that original result table and should not be interpreted as the leakage-controlled model-specific context selection added at the end of this notebook.


In [ ]:
# Fit and explain the best S2+Context model.
import warnings
warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", category=UserWarning, module="tqdm.auto")

try:
    import shap
except ImportError as exc:
    raise ImportError(
        "SHAP is not installed in this kernel. Install it with: python -m pip install shap"
    ) from exc

S2_CONTEXT_EXPLAIN_FEATURE_SET_NAME = "S2 selected + Context selected"
S2_CONTEXT_EXPLAIN_SPLIT = "test"

if S2_CONTEXT_EXPLAIN_FEATURE_SET_NAME not in FEATURE_SETS:
    raise KeyError(f"Unknown feature set: {S2_CONTEXT_EXPLAIN_FEATURE_SET_NAME}")

if "best_by_feature" in globals() and not best_by_feature.empty:
    s2_context_best_row = best_by_feature.loc[
        best_by_feature["feature_set"].eq(S2_CONTEXT_EXPLAIN_FEATURE_SET_NAME)
    ].iloc[0]
    S2_CONTEXT_EXPLAIN_MODEL_NAME = s2_context_best_row["model"]
else:
    s2_context_candidate = results.loc[
        results["feature_set"].eq(S2_CONTEXT_EXPLAIN_FEATURE_SET_NAME)
    ].sort_values(["test_original_r2", "test_log_r2"], ascending=False).iloc[0]
    S2_CONTEXT_EXPLAIN_MODEL_NAME = s2_context_candidate["model"]

s2_context_models = make_models(RANDOM_STATE)
if S2_CONTEXT_EXPLAIN_MODEL_NAME not in s2_context_models:
    raise KeyError(f"Unknown model: {S2_CONTEXT_EXPLAIN_MODEL_NAME}")

s2_context_feature_cols = FEATURE_SETS[S2_CONTEXT_EXPLAIN_FEATURE_SET_NAME]
s2_context_pipeline = s2_context_models[S2_CONTEXT_EXPLAIN_MODEL_NAME]
s2_context_pipeline.fit(train_df[s2_context_feature_cols], train_df[TARGET_LOG_COL])

X_train_s2_context_raw = train_df[s2_context_feature_cols].copy()
X_test_s2_context_raw = test_df[s2_context_feature_cols].copy()
X_explain_s2_context_raw = X_test_s2_context_raw if S2_CONTEXT_EXPLAIN_SPLIT == "test" else X_train_s2_context_raw
explain_s2_context_df = test_df if S2_CONTEXT_EXPLAIN_SPLIT == "test" else train_df

s2_context_preprocess = Pipeline(s2_context_pipeline.steps[:-1])
s2_context_final_model = s2_context_pipeline.steps[-1][1]

X_train_s2_context = pd.DataFrame(
    s2_context_preprocess.transform(X_train_s2_context_raw),
    columns=s2_context_feature_cols,
    index=X_train_s2_context_raw.index,
)
X_explain_s2_context = pd.DataFrame(
    s2_context_preprocess.transform(X_explain_s2_context_raw),
    columns=s2_context_feature_cols,
    index=X_explain_s2_context_raw.index,
)

if not hasattr(s2_context_final_model, "coef_"):
    raise TypeError(
        f"{S2_CONTEXT_EXPLAIN_MODEL_NAME} is not a linear coefficient model. "
        "Use TreeExplainer/PermutationExplainer instead."
    )

pred_test_s2_context_log = np.asarray(s2_context_pipeline.predict(X_test_s2_context_raw)).ravel()
s2_context_metrics = regression_metrics(test_df[TARGET_LOG_COL], pred_test_s2_context_log, test_df[TARGET_COL])
s2_context_pred_original = np.expm1(np.clip(pred_test_s2_context_log, 0, None))

s2_context_summary = pd.DataFrame([{
    "feature_set": S2_CONTEXT_EXPLAIN_FEATURE_SET_NAME,
    "model": S2_CONTEXT_EXPLAIN_MODEL_NAME,
    "explained_split": S2_CONTEXT_EXPLAIN_SPLIT,
    "n_features": len(s2_context_feature_cols),
    "n_train": len(train_df),
    "n_test": len(test_df),
    "test_original_r2": s2_context_metrics["original_r2"],
    "test_original_rmse": s2_context_metrics["original_rmse"],
    "test_log_r2": s2_context_metrics["log_r2"],
    "test_log_rmse": s2_context_metrics["log_rmse"],
    "test_spearman_r": s2_context_metrics["spearman_r"],
    "test_bias": s2_context_metrics["bias"],
}])

display(s2_context_summary.round(4))


In [ ]:
# S2+Context model coefficients.
s2_context_coef = np.asarray(s2_context_final_model.coef_, dtype=float).ravel()
s2_context_intercept = float(np.asarray(s2_context_final_model.intercept_).ravel()[0])

def feature_group(feature_name):
    return "S2" if feature_name in S2_SELECTED else "Context"

s2_context_coef_table = (
    pd.DataFrame({
        "feature": s2_context_feature_cols,
        "feature_group": [feature_group(c) for c in s2_context_feature_cols],
        "coefficient_log1p_standardized": s2_context_coef,
        "abs_coefficient": np.abs(s2_context_coef),
        "direction": np.where(s2_context_coef >= 0, "pushes prediction up", "pushes prediction down"),
    })
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

print(f"{S2_CONTEXT_EXPLAIN_MODEL_NAME} equation on standardized features: log1p(Turbidity) = {s2_context_intercept:.4f} + sum(coef_j * z_j)")
print("z_j = median-imputed and standardized feature value.")
display(s2_context_coef_table.round(5))

fig, ax = plt.subplots(figsize=(7, 7))
plot_df = s2_context_coef_table.head(20).sort_values("coefficient_log1p_standardized")
colors = np.where(plot_df["feature_group"].eq("S2"), "#4C78A8", "#F58518")
ax.barh(plot_df["feature"], plot_df["coefficient_log1p_standardized"], color=colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Coefficient on log1p(Turbidity), standardized features")
ax.set_title(f"Top coefficients | {S2_CONTEXT_EXPLAIN_MODEL_NAME} | S2+Context")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# Linear SHAP for the best S2+Context model.
s2_context_shap_explainer = shap.LinearExplainer(s2_context_final_model, X_train_s2_context)
s2_context_shap_explanation = s2_context_shap_explainer(X_explain_s2_context)
s2_context_shap_values = np.asarray(s2_context_shap_explanation.values)

s2_context_shap_importance = (
    pd.DataFrame({
        "feature": s2_context_feature_cols,
        "feature_group": [feature_group(c) for c in s2_context_feature_cols],
        "coefficient_log1p_standardized": s2_context_coef,
        "mean_abs_shap_log1p": np.abs(s2_context_shap_values).mean(axis=0),
        "mean_shap_log1p": s2_context_shap_values.mean(axis=0),
    })
    .sort_values("mean_abs_shap_log1p", ascending=False)
    .reset_index(drop=True)
)

s2_context_group_shap = (
    s2_context_shap_importance
    .groupby("feature_group", as_index=False)
    .agg(
        total_mean_abs_shap_log1p=("mean_abs_shap_log1p", "sum"),
        mean_abs_shap_log1p=("mean_abs_shap_log1p", "mean"),
        n_features=("feature", "count"),
    )
    .sort_values("total_mean_abs_shap_log1p", ascending=False)
)

print("Linear SHAP importance for the best S2+Context model.")
print("Larger mean_abs_shap_log1p = stronger average contribution to log1p(Turbidity).")
display(s2_context_shap_importance.round(5))

print("Group-level SHAP contribution: S2 vs Context.")
display(s2_context_group_shap.round(5))

fig, ax = plt.subplots(figsize=(7, 7))
plot_df = s2_context_shap_importance.head(20).sort_values("mean_abs_shap_log1p")
colors = np.where(plot_df["feature_group"].eq("S2"), "#4C78A8", "#F58518")
ax.barh(plot_df["feature"], plot_df["mean_abs_shap_log1p"], color=colors)
ax.set_xlabel("Mean |SHAP value| on log1p(Turbidity)")
ax.set_title(f"Linear SHAP global importance | {S2_CONTEXT_EXPLAIN_MODEL_NAME} | S2+Context")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

shap.summary_plot(
    s2_context_shap_values,
    X_explain_s2_context,
    feature_names=s2_context_feature_cols,
    max_display=min(20, len(s2_context_feature_cols)),
    show=False,
)
plt.title(f"Linear SHAP summary | {S2_CONTEXT_EXPLAIN_MODEL_NAME} | S2+Context")
plt.tight_layout()
plt.show()


In [ ]:
# Local SHAP explanation for the highest-observed turbidity sample in the S2+Context explained split.
pred_log_s2_context_explain = np.asarray(s2_context_pipeline.predict(X_explain_s2_context_raw)).ravel()
pred_original_s2_context_explain = np.expm1(np.clip(pred_log_s2_context_explain, 0, None))

local_s2_context_meta = explain_s2_context_df.loc[X_explain_s2_context_raw.index, [
    c for c in ["sample_id", "station_canonical", "year_be", "field_date", "sat_date", "abs_date_diff_days", TARGET_COL]
    if c in explain_s2_context_df.columns
]].copy()
local_s2_context_meta["predicted_turbidity"] = pred_original_s2_context_explain
local_s2_context_meta["residual"] = local_s2_context_meta["predicted_turbidity"] - local_s2_context_meta[TARGET_COL]

sample_pos = int(np.nanargmax(local_s2_context_meta[TARGET_COL].to_numpy(dtype=float)))
print("Local S2+Context SHAP example: highest observed turbidity in explained split")
display(local_s2_context_meta.iloc[[sample_pos]].round(4))

local_contrib = pd.DataFrame({
    "feature": s2_context_feature_cols,
    "feature_group": [feature_group(c) for c in s2_context_feature_cols],
    "shap_log1p": s2_context_shap_values[sample_pos],
})
local_group_contrib = (
    local_contrib.groupby("feature_group", as_index=False)
    .agg(total_shap_log1p=("shap_log1p", "sum"), total_abs_shap_log1p=("shap_log1p", lambda x: np.abs(x).sum()))
    .sort_values("total_abs_shap_log1p", ascending=False)
)
print("Local group contribution for this sample.")
display(local_group_contrib.round(5))

shap.plots.waterfall(s2_context_shap_explanation[sample_pos], max_display=min(15, len(s2_context_feature_cols)), show=False)
plt.title("Local S2+Context SHAP waterfall on log1p(Turbidity)")
plt.tight_layout()
plt.show()


## Reporting Summary: Requested Model Families

This table reports the same within-1-day temporal experiment for the requested model families. The `LinearSVR` row is the best `C`/`epsilon` configuration within each feature set; all other families use their configured model in this notebook. Metrics on the original scale are calculated after applying `expm1` to predictions trained on `log1p(Turbidity)`.


In [ ]:
# Summary table for the models reported in the methodology.
# Select the best LinearSVR hyperparameter setting within each feature set,
# while retaining one directly comparable result for every other requested family.
def report_model_family(model_name):
    model_name = str(model_name)
    if model_name.startswith("LinearSVR_"):
        return "LinearSVR"
    if model_name.startswith("PLS_"):
        return "PLS"
    return model_name


REPORT_MODEL_FAMILIES = [
    "Ridge",
    "ElasticNet",
    "HuberRegressor",
    "PLS",
    "LinearSVR",
    "RandomForest",
    "ExtraTrees",
    "XGBoost",
    "CatBoost",
]

tur_report_candidates = results.copy()
tur_report_candidates["model_family"] = tur_report_candidates["model"].map(report_model_family)
tur_report_candidates = tur_report_candidates.loc[
    tur_report_candidates["model_family"].isin(REPORT_MODEL_FAMILIES)
].copy()

tur_report_selected = (
    tur_report_candidates
    .sort_values(
        ["feature_set", "model_family", "test_original_r2", "test_log_r2", "test_original_rmse"],
        ascending=[True, True, False, False, True],
    )
    .groupby(["feature_set", "model_family"], as_index=False)
    .head(1)
    .copy()
)

tur_model_summary = (
    tur_report_selected
    .rename(columns={
        "feature_set": "feature set",
        "test_original_r2": "R2",
        "test_original_rmse": "RMSE",
        "test_original_mae": "MAE",
        "test_log_r2": "log R2",
        "test_log_rmse": "log RMSE",
        "test_spearman_r": "spearman r",
        "test_bias": "bias",
    })
    [["model", "feature set", "R2", "RMSE", "MAE", "log R2", "log RMSE", "spearman r", "bias"]]
    .sort_values(["R2", "log R2"], ascending=False)
    .reset_index(drop=True)
)

print(
    "Within-1-day temporal test: train 2562-2566, test 2567-2568. "
    "Metrics are evaluated on the held-out test set."
)
display(tur_model_summary.round(4))

best_tur_model = tur_model_summary.iloc[[0]].copy()
print("Best model by original-scale R2")
display(best_tur_model.round(4))


## Model-Specific Context Group Selection (Training-Only)

This experiment evaluates models other than LinearSVR using five paired feature schemes: `S2 + rainfall`, `S2 + runoff`, `S2 + meteorology`, `S2 + static water`, and `S2 + selected combined context`.

The candidate pools are intentionally compact. They represent short, event-scale, and antecedent conditions without including every 1-720-hour variable. For each model, candidate subsets are ranked using pooled leave-one-year-out predictions from the training years 2562-2566. The held-out years 2567-2568 are not used to select context features or context-group combinations.

The combined-context search is hierarchical: it first locks the best training-CV subset from each physical group, then evaluates all non-empty combinations of those four group winners. This keeps the search interpretable and computationally appropriate for the small sample size.


In [ ]:
# Compact, domain-guided context candidates and non-LinearSVR model families.
from itertools import combinations

from sklearn.base import clone
from sklearn.model_selection import LeaveOneGroupOut

CONTEXT_SUBSET_CANDIDATES = {
    "rainfall": {
        "rain_6h": ["rain_gsmap_prior_6h_mm"],
        "rain_24h": ["rain_gsmap_prior_24h_mm"],
        "rain_72h": ["rain_gsmap_prior_72h_mm"],
        "rain_168h": ["rain_gsmap_prior_168h_mm"],
        "rain_24h_intensity": [
            "rain_gsmap_prior_24h_mm",
            "rain_gsmap_max_prior_24h_mm_h",
        ],
        "rain_24h_72h": [
            "rain_gsmap_prior_24h_mm",
            "rain_gsmap_prior_72h_mm",
        ],
        "rain_short_long": [
            "rain_gsmap_prior_6h_mm",
            "rain_gsmap_prior_24h_mm",
            "rain_gsmap_prior_168h_mm",
        ],
        "rain_state": [
            "rain_gsmap_prior_24h_mm",
            "hours_since_last_rain_gsmap_mean",
        ],
    },
    "runoff": {
        "runoff_6h": ["era5h_runoff_prior_6h_mm"],
        "runoff_24h": ["era5h_runoff_prior_24h_mm"],
        "runoff_72h": ["era5h_runoff_prior_72h_mm"],
        "runoff_168h": ["era5h_runoff_prior_168h_mm"],
        "runoff_6h_24h": [
            "era5h_runoff_prior_6h_mm",
            "era5h_runoff_prior_24h_mm",
        ],
        "runoff_24h_72h": [
            "era5h_runoff_prior_24h_mm",
            "era5h_runoff_prior_72h_mm",
        ],
        "runoff_short_long": [
            "era5h_runoff_prior_24h_mm",
            "era5h_runoff_prior_168h_mm",
        ],
    },
    "meteorology": {
        "air_temp_24h": ["era5h_air_temp_prior_24h_C"],
        "dewpoint_24h": ["era5h_dewpoint_prior_24h_C"],
        "wind_mean_24h": ["era5h_wind_mean_prior_24h_m_s"],
        "wind_max_24h": ["era5h_wind_max_prior_24h_m_s"],
        "solar_24h": ["era5h_solar_prior_24h_MJ_m2"],
        "temp_dewpoint_24h": [
            "era5h_air_temp_prior_24h_C",
            "era5h_dewpoint_prior_24h_C",
        ],
        "wind_24h": [
            "era5h_wind_mean_prior_24h_m_s",
            "era5h_wind_max_prior_24h_m_s",
        ],
        "meteorology_compact": [
            "era5h_air_temp_prior_24h_C",
            "era5h_dewpoint_prior_24h_C",
            "era5h_wind_mean_prior_24h_m_s",
            "era5h_solar_prior_24h_MJ_m2",
        ],
    },
    "static water": {
        "jrc_occurrence": ["jrc_occurrence_mean_500m"],
        "jrc_seasonality": ["jrc_seasonality_mean_500m"],
        "jrc_recurrence": ["jrc_recurrence_mean_500m"],
        "wc_water_500m": ["wc_water_frac_500m"],
        "wc_water_1000m": ["wc_water_frac_1000m"],
        "wc_water_5000m": ["wc_water_frac_5000m"],
        "jrc_profile": [
            "jrc_occurrence_mean_500m",
            "jrc_seasonality_mean_500m",
            "jrc_recurrence_mean_500m",
        ],
        "wc_water_multiscale": [
            "wc_water_frac_500m",
            "wc_water_frac_1000m",
            "wc_water_frac_5000m",
        ],
        "static_water_compact": [
            "jrc_occurrence_mean_500m",
            "jrc_recurrence_mean_500m",
            "wc_water_frac_5000m",
        ],
    },
}

context_selection_models = make_models(RANDOM_STATE)
CONTEXT_MODEL_NAMES = [
    name for name in [
        "Ridge", "ElasticNet", "BayesianRidge", "HuberRegressor", "PLS_2",
        "RandomForest", "ExtraTrees", "HistGradientBoosting", "XGBoost", "CatBoost",
    ]
    if name in context_selection_models
]

candidate_context_columns = sorted({
    column
    for subsets in CONTEXT_SUBSET_CANDIDATES.values()
    for columns in subsets.values()
    for column in columns
})
missing_context_candidates = [
    column for column in candidate_context_columns if column not in train_df.columns
]
if missing_context_candidates:
    raise KeyError(f"Missing context candidate columns: {missing_context_candidates}")

candidate_audit = pd.DataFrame([
    {
        "context_group": group_name,
        "candidate_subset": subset_name,
        "n_context_features": len(columns),
        "context_features": ", ".join(columns),
    }
    for group_name, subsets in CONTEXT_SUBSET_CANDIDATES.items()
    for subset_name, columns in subsets.items()
])

print("Models:", CONTEXT_MODEL_NAMES)
print("Unique compact context candidates:", len(candidate_context_columns))
display(candidate_audit)


In [ ]:
# Training-only pooled leave-one-year-out scoring and held-out test evaluation.
selection_cache = {}


def unique_columns(columns):
    return list(dict.fromkeys(columns))


def training_oof_metrics(model_name, context_columns):
    context_columns = unique_columns(context_columns)
    cache_key = (model_name, tuple(context_columns))
    if cache_key in selection_cache:
        return selection_cache[cache_key].copy()

    feature_columns = S2_SELECTED + context_columns
    X = train_df[feature_columns].copy()
    y_log = train_df[TARGET_LOG_COL].to_numpy(dtype=float)
    y_original = train_df[TARGET_COL].to_numpy(dtype=float)
    groups = pd.to_numeric(train_df["year_be"], errors="coerce").to_numpy()

    logo = LeaveOneGroupOut()
    oof_pred_log = np.full(len(train_df), np.nan, dtype=float)
    fold_count = 0

    for fit_index, valid_index in logo.split(X, y_log, groups=groups):
        estimator = clone(context_selection_models[model_name])
        estimator.fit(X.iloc[fit_index], y_log[fit_index])
        oof_pred_log[valid_index] = np.asarray(
            estimator.predict(X.iloc[valid_index])
        ).ravel()
        fold_count += 1

    valid = np.isfinite(oof_pred_log)
    metrics = regression_metrics(
        y_log[valid],
        oof_pred_log[valid],
        y_original[valid],
    )
    result = {
        "n_cv": int(valid.sum()),
        "n_cv_folds": int(fold_count),
        **{f"cv_{name}": value for name, value in metrics.items()},
    }
    selection_cache[cache_key] = result.copy()
    return result


def holdout_metrics(model_name, context_columns):
    context_columns = unique_columns(context_columns)
    feature_columns = S2_SELECTED + context_columns
    estimator = clone(context_selection_models[model_name])
    estimator.fit(train_df[feature_columns], train_df[TARGET_LOG_COL])
    pred_test_log = np.asarray(
        estimator.predict(test_df[feature_columns])
    ).ravel()
    metrics = regression_metrics(
        test_df[TARGET_LOG_COL],
        pred_test_log,
        test_df[TARGET_COL],
    )
    return {
        "n_train": len(train_df),
        "n_test": len(test_df),
        **{f"test_{name}": value for name, value in metrics.items()},
    }


def cv_sort(frame, group_columns):
    return frame.sort_values(
        group_columns + ["cv_original_r2", "cv_log_r2", "cv_original_rmse"],
        ascending=[True] * len(group_columns) + [False, False, True],
    )


In [ ]:
# Stage 1: select the best compact subset inside each physical context group.
baseline_cv_rows = []
group_cv_rows = []

for model_name in CONTEXT_MODEL_NAMES:
    baseline_cv_rows.append({
        "model": model_name,
        "scheme": "S2 only",
        "context_group": "none",
        "candidate_subset": "none",
        "context_features": [],
        "n_context_features": 0,
        **training_oof_metrics(model_name, []),
    })

    for group_name, subset_definitions in CONTEXT_SUBSET_CANDIDATES.items():
        for subset_name, context_columns in subset_definitions.items():
            group_cv_rows.append({
                "model": model_name,
                "scheme": f"S2 + {group_name}",
                "context_group": group_name,
                "candidate_subset": subset_name,
                "context_features": list(context_columns),
                "n_context_features": len(context_columns),
                **training_oof_metrics(model_name, context_columns),
            })

baseline_cv_results = pd.DataFrame(baseline_cv_rows)
context_group_cv_results = pd.DataFrame(group_cv_rows)

best_context_subset_by_model_group = (
    cv_sort(context_group_cv_results, ["model", "context_group"])
    .groupby(["model", "context_group"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

baseline_r2_lookup = baseline_cv_results.set_index("model")["cv_original_r2"]
best_context_subset_by_model_group["cv_delta_r2_vs_s2"] = (
    best_context_subset_by_model_group["cv_original_r2"]
    - best_context_subset_by_model_group["model"].map(baseline_r2_lookup)
)

print("Best training-CV subset inside each physical context group")
display(
    best_context_subset_by_model_group[[
        "model", "scheme", "candidate_subset", "n_context_features",
        "context_features", "cv_original_r2", "cv_original_rmse",
        "cv_log_r2", "cv_spearman_r", "cv_delta_r2_vs_s2",
    ]].round(4)
)


In [ ]:
# Stage 2: test all combinations of the four model-specific group winners in training CV.
combined_cv_rows = []

for model_name in CONTEXT_MODEL_NAMES:
    model_group_winners = (
        best_context_subset_by_model_group.loc[
            best_context_subset_by_model_group["model"].eq(model_name)
        ]
        .sort_values("context_group")
        .to_dict("records")
    )

    for n_groups in range(1, len(model_group_winners) + 1):
        for selected_groups in combinations(model_group_winners, n_groups):
            group_names = [row["context_group"] for row in selected_groups]
            subset_names = [row["candidate_subset"] for row in selected_groups]
            context_columns = unique_columns([
                column
                for row in selected_groups
                for column in row["context_features"]
            ])
            combined_cv_rows.append({
                "model": model_name,
                "scheme": "S2 + selected combined context",
                "context_group": " + ".join(group_names),
                "candidate_subset": " + ".join(subset_names),
                "context_features": context_columns,
                "n_context_features": len(context_columns),
                **training_oof_metrics(model_name, context_columns),
            })

combined_context_cv_results = pd.DataFrame(combined_cv_rows)
best_combined_context_by_model = (
    cv_sort(combined_context_cv_results, ["model"])
    .groupby("model", as_index=False)
    .head(1)
    .reset_index(drop=True)
)
best_combined_context_by_model["cv_delta_r2_vs_s2"] = (
    best_combined_context_by_model["cv_original_r2"]
    - best_combined_context_by_model["model"].map(baseline_r2_lookup)
)

print("Best combined-context configuration selected without the held-out test years")
display(
    best_combined_context_by_model[[
        "model", "context_group", "candidate_subset", "n_context_features",
        "context_features", "cv_original_r2", "cv_original_rmse",
        "cv_log_r2", "cv_spearman_r", "cv_delta_r2_vs_s2",
    ]].round(4)
)


In [ ]:
# Evaluate only the locked training-CV selections on held-out years 2567-2568.
locked_cv_candidates = pd.concat([
    baseline_cv_results,
    best_context_subset_by_model_group,
    best_combined_context_by_model,
], ignore_index=True, sort=False)

holdout_rows = []
for row in locked_cv_candidates.to_dict("records"):
    holdout_rows.append({
        "model": row["model"],
        "scheme": row["scheme"],
        "candidate_subset": row["candidate_subset"],
        "context_features": row["context_features"],
        "n_context_features": row["n_context_features"],
        "cv_original_r2": row["cv_original_r2"],
        "cv_original_rmse": row["cv_original_rmse"],
        "cv_log_r2": row["cv_log_r2"],
        **holdout_metrics(row["model"], row["context_features"]),
    })

context_selection_test_results = pd.DataFrame(holdout_rows)
test_s2_r2_lookup = (
    context_selection_test_results.loc[
        context_selection_test_results["scheme"].eq("S2 only")
    ]
    .set_index("model")["test_original_r2"]
)
context_selection_test_results["test_delta_r2_vs_s2"] = (
    context_selection_test_results["test_original_r2"]
    - context_selection_test_results["model"].map(test_s2_r2_lookup)
)

scheme_order = {
    "S2 only": 0,
    "S2 + rainfall": 1,
    "S2 + runoff": 2,
    "S2 + meteorology": 3,
    "S2 + static water": 4,
    "S2 + selected combined context": 5,
}
context_selection_test_results["scheme_order"] = (
    context_selection_test_results["scheme"].map(scheme_order)
)
context_selection_test_results = context_selection_test_results.sort_values(
    ["model", "scheme_order"]
).reset_index(drop=True)

print("Locked paired test results; feature selection used training years only")
display(
    context_selection_test_results[[
        "model", "scheme", "candidate_subset", "n_context_features",
        "context_features", "n_train", "n_test",
        "cv_original_r2", "test_original_r2", "test_delta_r2_vs_s2",
        "test_original_rmse", "test_original_mae", "test_log_r2",
        "test_log_rmse", "test_spearman_r", "test_bias",
    ]].round(4)
)

# Lock one final scheme per model by training-CV R2, not by held-out test R2.
locked_best_by_model = (
    cv_sort(locked_cv_candidates, ["model"])
    .groupby("model", as_index=False)
    .head(1)
    .reset_index(drop=True)
)
locked_best_by_model = locked_best_by_model.merge(
    context_selection_test_results,
    on=["model", "scheme", "candidate_subset"],
    how="left",
    suffixes=("_selected", ""),
)

print("Final model-specific feature scheme locked by training-only CV")
display(
    locked_best_by_model[[
        "model", "scheme", "candidate_subset", "n_context_features",
        "context_features", "cv_original_r2", "test_original_r2",
        "test_delta_r2_vs_s2", "test_original_rmse", "test_original_mae",
        "test_log_r2", "test_spearman_r", "test_bias",
    ]].round(4)
)

MODEL_SPECIFIC_COMBINED_CONTEXT = {
    row["model"]: list(row["context_features"])
    for row in best_combined_context_by_model.to_dict("records")
}
MODEL_SPECIFIC_LOCKED_FEATURES = {
    row["model"]: S2_SELECTED + list(row["context_features"])
    for row in locked_best_by_model.to_dict("records")
}


In [ ]:
# Paired held-out R2 change relative to the same model using S2 only.
plot_schemes = [
    "S2 + rainfall",
    "S2 + runoff",
    "S2 + meteorology",
    "S2 + static water",
    "S2 + selected combined context",
]
delta_matrix = (
    context_selection_test_results.loc[
        context_selection_test_results["scheme"].isin(plot_schemes)
    ]
    .pivot(index="model", columns="scheme", values="test_delta_r2_vs_s2")
    .reindex(columns=plot_schemes)
)

fig, ax = plt.subplots(figsize=(12, max(5, 0.55 * len(delta_matrix))))
image = ax.imshow(delta_matrix.to_numpy(dtype=float), cmap="RdBu_r", vmin=-0.4, vmax=0.4, aspect="auto")
ax.set_xticks(np.arange(len(delta_matrix.columns)))
ax.set_xticklabels(delta_matrix.columns, rotation=25, ha="right")
ax.set_yticks(np.arange(len(delta_matrix.index)))
ax.set_yticklabels(delta_matrix.index)
ax.set_title("Held-out change in original-scale R2 relative to S2 only")

for row_index in range(delta_matrix.shape[0]):
    for column_index in range(delta_matrix.shape[1]):
        value = delta_matrix.iloc[row_index, column_index]
        if np.isfinite(value):
            ax.text(column_index, row_index, f"{value:+.3f}", ha="center", va="center", fontsize=8)

fig.colorbar(image, ax=ax, label="Delta R2 (S2 + context minus S2 only)")
plt.tight_layout()
plt.show()


### Interpretation Rule

Use `locked_best_by_model` for the model-specific feature conclusion because its scheme was selected from leave-one-year-out predictions within 2562-2566. The held-out 2567-2568 metrics describe generalization after selection.

An improvement for one model but not others should be reported as a **model-dependent context contribution**. Do not choose a different context combination after inspecting held-out test R2. The original fixed-context and LinearSVR sections are retained as historical baselines, while this section provides the leakage-controlled comparison for the other model families.


## Requested Fixed-Context Reporting Override

For the reporting table below, Ridge, ElasticNet, PLS, and CatBoost are returned to the original fixed four-feature context set because that configuration had the better held-out R2 in the original paired experiment. The other non-LinearSVR models retain their model-specific training-CV selections, and LinearSVR retains its original locked four-feature context set.

This hybrid table is an **exploratory reporting comparison**, not a fully training-only model-selection result. These overrides were chosen after comparing held-out test performance. Therefore, use `locked_best_by_model` when a leakage-controlled selection claim is required, and use `final_turbidity_report_table` only when explicitly describing the fixed-context/model-specific hybrid comparison.


In [ ]:
# Build the requested final comparison with three fixed-context overrides.
REPORTING_CONTEXT_OVERRIDES = {
    "Ridge": list(CONTEXT_SELECTED),
    "ElasticNet": list(CONTEXT_SELECTED),
    "PLS_2": list(CONTEXT_SELECTED),
    "CatBoost": list(CONTEXT_SELECTED),
}

REPORTING_MODELS = [
    ("Regularized linear", "Ridge", "Ridge"),
    ("Regularized linear", "Elastic Net", "ElasticNet"),
    ("Robust linear", "Huber Regression", "HuberRegressor"),
    ("Latent-variable", "PLS", "PLS_2"),
    ("Support-vector", "LinearSVR", "LinearSVR_C0.5_eps0.0"),
    ("Tree ensemble", "Random Forest", "RandomForest"),
    ("Tree ensemble", "Extra Trees", "ExtraTrees"),
    ("Gradient-boosted tree", "XGBoost", "XGBoost"),
    ("Gradient-boosted tree", "CatBoost", "CatBoost"),
]

locked_context_lookup = {
    row["model"]: list(row["context_features"])
    for row in locked_best_by_model.to_dict("records")
}

report_rows = []
FINAL_REPORTING_FEATURES = {}

for model_family, display_model, model_name in REPORTING_MODELS:
    if model_name.startswith("LinearSVR_"):
        context_columns = list(CONTEXT_SELECTED)
        selection_source = "Original locked fixed context"
    elif model_name in REPORTING_CONTEXT_OVERRIDES:
        context_columns = list(REPORTING_CONTEXT_OVERRIDES[model_name])
        selection_source = "Requested original fixed-context override"
    else:
        context_columns = list(locked_context_lookup[model_name])
        selection_source = "Training-only LOYO-CV selection"

    s2_metrics = holdout_metrics(model_name, [])
    context_metrics = holdout_metrics(model_name, context_columns)
    FINAL_REPORTING_FEATURES[model_name] = S2_SELECTED + context_columns

    report_rows.append({
        "Model family": model_family,
        "Model": display_model,
        "S2 only R2": s2_metrics["test_original_r2"],
        "S2 only RMSE": s2_metrics["test_original_rmse"],
        "S2 only MAE": s2_metrics["test_original_mae"],
        "S2 + Context R2": context_metrics["test_original_r2"],
        "S2 + Context RMSE": context_metrics["test_original_rmse"],
        "S2 + Context MAE": context_metrics["test_original_mae"],
        "Context selection source": selection_source,
        "Context features": context_columns,
    })

final_turbidity_report_table = pd.DataFrame(report_rows)

print("Requested final Turbidity model comparison")
display(
    final_turbidity_report_table[[
        "Model family", "Model",
        "S2 only R2", "S2 only RMSE", "S2 only MAE",
        "S2 + Context R2", "S2 + Context RMSE", "S2 + Context MAE",
    ]].round(4)
)

print("Context provenance for each reported model")
display(
    final_turbidity_report_table[[
        "Model", "Context selection source", "Context features",
    ]]
)
